# Bounding General Observables with the NPA Method

This notebook shows how to bound **any observable** in the ground state, not just the energy. We combine SDP lower bounds with variational upper bounds (DMRG) to sandwich ground-state expectation values.

| Step | What you'll see |
|------|----------------|
| **1** | Setup & imports |
| **2** | Theory: bounding observables via energy constraints |
| **3** | Ground-state energy bounds (SDP lower + DMRG upper) |
| **4** | Bounding a single observable |
| **5** | Examples: magnetisation, correlations, structure factor |
| **6** | Scaling with system size |
| **7** | Persistence with the ArtifactManager |

---
## 1. Setup & Imports

In [14]:
import sys, time, shutil
import numpy as np
from pathlib import Path

PROJECT_ROOT = str(Path("..").resolve())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# --- Spin models ---
from spins.models import (
    heisenberg_hamiltonian_exact,
    heisenberg_hamiltonian_dict,
    magnetization_z_dict,
    staggered_magnetization_z_dict,
    nearest_neighbor_correlation_dict,
    two_point_correlation_dict,
    single_site_pauli_dict,
)

# --- Pauli algebra & basis ---
from spins.pauli_logic import PauliWord, compile_moment_matrix_rep
from spins.basis_builder import generate_npa_basis, generate_heisenberg_paper_basis

# --- Symmetries ---
from spins.symmetry import SymmetryManager

# --- SDP layer ---
from spins.spins_sdp import (
    build_block_reps,
    build_block_diagonal_sdp,
    compile_operator_linear_form,
    solve_pauli_relaxation,
    bound_observable,
)

# --- DMRG (variational upper bound) ---
from spins.variational import (
    build_heisenberg_pbc_model,
    initial_product_state,
    dmrg_upper_bound,
)

# --- Persistence ---
from artifact_manager import ArtifactManager, config_hash

print("All imports OK ✓")
print(f"Project root: {PROJECT_ROOT}")

All imports OK ✓
Project root: /users/eleves-b/2023/sherab-losel.matos/thesis/SpinsSDP/SpinsSDP


---
## 2. Theory: Bounding Observables via Energy Constraints

Suppose we know that the ground-state energy satisfies:
$$E_L \;\leq\; E_0 \;\leq\; E_U$$

where $E_L$ comes from an SDP relaxation and $E_U$ from a variational method (e.g. DMRG).

To bound an observable $\hat{O}$ in the ground state, we solve two SDPs:

$$\min / \max \;\; \langle O \rangle$$
$$\text{s.t.} \quad \Gamma \succeq 0, \quad \langle I \rangle = 1, \quad E_L \leq \langle H \rangle \leq E_U$$

This yields **certified bounds** on $\langle O \rangle_{\text{g.s.}}$ that are valid for any quantum state whose energy falls in $[E_L, E_U]$.

The function `bound_observable(...)` does exactly this: it solves both the min and max SDPs and returns the interval $[\text{lb}, \text{ub}]$.

---
## 3. Ground-State Energy Bounds

Before bounding observables, we need the **energy window** $[E_L, E_U]$.

- **$E_L$** (lower bound): from the moment-relaxation SDP.
- **$E_U$** (upper bound): from DMRG (or exact diagonalisation for small $N$).

We also compute the exact energy as a reference.

In [15]:
# --- Parameters ---
N = 8
boundary = "periodic"

H_exact = heisenberg_hamiltonian_exact(N, boundary=boundary)
H_dict = heisenberg_hamiltonian_dict(N, boundary=boundary)
sym = SymmetryManager.default_for_heisenberg(N)

# --- Exact ground-state energy (reference) ---
t0 = time.perf_counter()
E0_exact = float(H_exact.eigenenergies(eigvals=1)[0])
dt_exact = time.perf_counter() - t0
print(f"Exact E0 (N={N}):  {E0_exact:.6f}  ({dt_exact:.3f}s)")

Exact E0 (N=8):  -3.651093  (0.009s)


In [16]:
# --- SDP lower bound ---
basis = generate_npa_basis(N, k=2)
t0 = time.perf_counter()
E_lb = solve_pauli_relaxation(
    basis=basis.words,
    operator=H_dict,
    symmetry_manager=sym,
    sense="min",
    mosek_tol=1e-7,
)
dt_sdp = time.perf_counter() - t0
print(f"SDP lower bound:   {E_lb:.6f}  (NPA k=2, {len(basis.words)} words, {dt_sdp:.3f}s)")

SDP lower bound:   -3.651708  (NPA k=2, 277 words, 0.162s)


In [17]:
# --- DMRG upper bound ---
model = build_heisenberg_pbc_model(N)
psi = initial_product_state(model, kind="neel")

t0 = time.perf_counter()
E_ub, psi_out, info, dt_dmrg = dmrg_upper_bound(
    model, psi,
    chi_max=128,
    svd_min=1e-10,
    max_E_err=1e-10,
    mixer=True,
    dchi=32,
    nsweeps=2,
    combine=True,
)
print(f"DMRG upper bound:  {E_ub:.6f}  (chi_max=128, {dt_dmrg:.3f}s)")

print(f"\n{'='*50}")
print(f"  Energy window:  [{E_lb:.6f}, {E_ub:.6f}]")
print(f"  Exact E0:        {E0_exact:.6f}")
print(f"  Window width:    {E_ub - E_lb:.6f}")
print(f"{'='*50}")

/users/eleves-b/2023/sherab-losel.matos/thesis/SpinsSDP/SpinsSDP/.venv/lib/python3.13/site-packages/tenpy/networks/mps.py:1629: UserWarning: unit_cell_width is a new argument for MPS and similar classes. It is optional for now, but will become mandatory in a future release. The default value (unit_cell_width=len(sites)) is correct, iff the lattice is a Chain. For other lattices, it is incorrect. It is used for dipolar charges and correlation_function2.
  super().__init__(sites, bc, unit_cell_width)


DMRG upper bound:  -3.651093  (chi_max=128, 0.835s)

  Energy window:  [-3.651708, -3.651093]
  Exact E0:        -3.651093
  Window width:    0.000615


---
## 4. Bounding a Single Observable

Let's bound the **nearest-neighbour Z-Z correlation**:

$$C_{zz} = \frac{1}{N}\sum_{\langle i,j\rangle} \sigma_i^z \sigma_j^z$$

This is a physical observable that characterises the magnetic ordering of the chain.

In [18]:
# Define the observable
obs_zz = nearest_neighbor_correlation_dict(N, axis="z", boundary=boundary)
print(f"Observable: nearest-neighbour ZZ correlation ({len(obs_zz)} Pauli terms)")
for word, coeff in list(obs_zz.items())[:4]:
    print(f"  {coeff:+.4f} · {word}")
print(f"  ...")

Observable: nearest-neighbour ZZ correlation (8 Pauli terms)
  +0.1250 · Z0Z1
  +0.1250 · Z1Z2
  +0.1250 · Z2Z3
  +0.1250 · Z3Z4
  ...


In [20]:
# Bound the observable using the energy window
t0 = time.perf_counter()
result = bound_observable(
    basis=basis.words,
    hamiltonian=H_dict,
    observable=obs_zz,
    energy_lb=E_lb,
    energy_ub=E_ub,
    symmetry_manager=sym,
    mosek_tol=1e-7,
)
dt = time.perf_counter() - t0

# Exact value for comparison
import qutip as qt
psi_gs = H_exact.groundstate()[1]

# Build the exact ZZ correlation operator
C_zz_exact_op = qt.qzero([2]*N)
for i in range(N):
    j = (i + 1) % N
    op_list = [qt.qeye(2)] * N
    op_list[i] = qt.sigmaz()
    op_list[j] = qt.sigmaz()
    C_zz_exact_op += (1.0 / N) * qt.tensor(op_list)
C_zz_exact = float(qt.expect(C_zz_exact_op, psi_gs))

print(f"Nearest-neighbour ZZ correlation (N={N}):")
print(f"  SDP upper bound:  {result.ub:.6f}")
print(f"  Exact value:       {C_zz_exact:.6f}")
print(f"  SDP lower bound:  {result.lb:.6f}")
print(f"  Bracket width:    {result.ub - result.lb:.6f}")
print(f"  Time:             {dt:.3f}s")

# Verify the exact value is within bounds
assert result.lb - 1e-4 <= C_zz_exact <= result.ub + 1e-4, "Exact value outside bounds!"
print(f"\n  ✓ Exact value lies within the certified bounds")

Nearest-neighbour ZZ correlation (N=8):
  SDP upper bound:  -0.603199
  Exact value:       -0.608516
  SDP lower bound:  -0.613696
  Bracket width:    0.010496
  Time:             0.328s

  ✓ Exact value lies within the certified bounds


---
## 5. Examples: Multiple Observables

Let's bound several physically interesting observables at once:
1. **Uniform magnetisation** $M_z = \frac{1}{N}\sum_i Z_i$
2. **Staggered magnetisation** $M_z^{\text{stag}} = \frac{1}{N}\sum_i (-1)^i Z_i$
3. **NN correlations** along each axis ($XX$, $YY$, $ZZ$)
4. **Two-point correlation** $\langle Z_0 Z_r \rangle$ at distance $r = N/2$

In [21]:
# Build all observables
observables = {
    "M_z (uniform)":  magnetization_z_dict(N),
    "M_z (staggered)": staggered_magnetization_z_dict(N),
    "C_xx (NN)":       nearest_neighbor_correlation_dict(N, axis="x", boundary=boundary),
    "C_yy (NN)":       nearest_neighbor_correlation_dict(N, axis="y", boundary=boundary),
    "C_zz (NN)":       nearest_neighbor_correlation_dict(N, axis="z", boundary=boundary),
    f"Z0·Z{N//2}":     two_point_correlation_dict(N, i=0, j=N//2, axis="z"),
}

print(f"Bounding {len(observables)} observables for N={N} Heisenberg chain (PBC)")
print(f"Energy window: [{E_lb:.6f}, {E_ub:.6f}]\n")

print(f"{'Observable':<20s}  {'Lower':>10s}  {'Upper':>10s}  {'Width':>10s}  {'Time (s)':>10s}")
print("-" * 68)

for name, obs in observables.items():
    t0 = time.perf_counter()
    res = bound_observable(
        basis=basis.words,
        hamiltonian=H_dict,
        observable=obs,
        energy_lb=E_lb,
        energy_ub=E_ub,
        symmetry_manager=sym,
        mosek_tol=1e-7,
    )
    dt = time.perf_counter() - t0
    print(f"{name:<20s}  {res.lb:>10.6f}  {res.ub:>10.6f}  {res.ub - res.lb:>10.6f}  {dt:>10.3f}")

Bounding 6 observables for N=8 Heisenberg chain (PBC)
Energy window: [-3.651708, -3.651093]

Observable                 Lower       Upper       Width    Time (s)
--------------------------------------------------------------------
M_z (uniform)           0.000000    0.000000    0.000000       0.273
M_z (staggered)         0.000000    0.000000    0.000000       0.264
C_xx (NN)              -0.613696   -0.603199    0.010496       0.296
C_yy (NN)              -0.619148   -0.598155    0.020993       0.297
C_zz (NN)              -0.613696   -0.603199    0.010496       0.297
Z0·Z4                   0.185250    0.212435    0.027184       0.292


In [22]:
# Verify all bounds against exact values
psi_gs = H_exact.groundstate()[1]

print(f"\nVerification against exact ground state:\n")
print(f"{'Observable':<20s}  {'Lower':>10s}  {'Exact':>10s}  {'Upper':>10s}  {'OK?':>5s}")
print("-" * 62)

def exact_expectation(N, obs_dict, psi_gs):
    """Compute exact expectation value from Pauli dict."""
    result = 0.0
    for word, coeff in obs_dict.items():
        # Build the full operator for this Pauli word
        op_list = [qt.qeye(2)] * N
        for i in range(N):
            bit = 1 << i
            has_x = (word.x_mask & bit) != 0
            has_z = (word.z_mask & bit) != 0
            if has_x and has_z:
                op_list[i] = qt.sigmay()
            elif has_x:
                op_list[i] = qt.sigmax()
            elif has_z:
                op_list[i] = qt.sigmaz()
        full_op = qt.tensor(op_list)
        result += float(coeff) * float(qt.expect(full_op, psi_gs).real)
    return result

for name, obs in observables.items():
    res = bound_observable(
        basis=basis.words,
        hamiltonian=H_dict,
        observable=obs,
        energy_lb=E_lb,
        energy_ub=E_ub,
        symmetry_manager=sym,
        mosek_tol=1e-7,
    )
    exact_val = exact_expectation(N, obs, psi_gs)
    ok = res.lb - 1e-4 <= exact_val <= res.ub + 1e-4
    print(f"{name:<20s}  {res.lb:>10.6f}  {exact_val:>10.6f}  {res.ub:>10.6f}  {'✓' if ok else '✗':>5s}")


Verification against exact ground state:

Observable                 Lower       Exact       Upper    OK?
--------------------------------------------------------------
M_z (uniform)           0.000000   -0.000000    0.000000      ✓
M_z (staggered)         0.000000   -0.000000    0.000000      ✓
C_xx (NN)              -0.613696   -0.608516   -0.603199      ✓
C_yy (NN)              -0.619148   -0.608516   -0.598155      ✓
C_zz (NN)              -0.613696   -0.608516   -0.603199      ✓
Z0·Z4                   0.185250    0.198831    0.212435      ✓


---
## 6. Scaling with System Size

Beyond exact diagonalisation, the real power of the SDP+DMRG approach is that it scales to large system sizes. Let's demonstrate how the energy window and observable bounds behave as $N$ grows. We use DMRG for the upper bound and the SDP for the lower bound.

For speed, we'll stay in a moderate range ($N = 4, 6, 8$).

In [23]:
# Energy window scaling
Ns = [4, 6, 8, 10, 12]
energy_data = {}

print(f"{'N':>3s}  {'E_lb (SDP)':>12s}  {'E0 (exact)':>12s}  {'E_ub (DMRG)':>12s}  {'Window':>10s}")
print("-" * 58)

for N in Ns:
    H_dict_N = heisenberg_hamiltonian_dict(N, boundary="periodic")
    sym_N = SymmetryManager.default_for_heisenberg(N)
    basis_N = generate_npa_basis(N, k=2)
    
    # SDP lower bound
    E_lb_N = solve_pauli_relaxation(
        basis=basis_N.words,
        operator=H_dict_N,
        symmetry_manager=sym_N,
        sense="min",
        mosek_tol=1e-7,
    )
    
    # DMRG upper bound
    model_N = build_heisenberg_pbc_model(N)
    psi_N = initial_product_state(model_N, kind="neel")
    E_ub_N, _, _, _ = dmrg_upper_bound(
        model_N, psi_N,
        chi_max=128, svd_min=1e-10, max_E_err=1e-10,
        mixer=True, dchi=32, nsweeps=2, combine=True,
    )
    
    # Exact
    H_exact_N = heisenberg_hamiltonian_exact(N, boundary="periodic")
    E0_N = float(H_exact_N.eigenenergies(eigvals=1)[0])
    
    energy_data[N] = {"E_lb": E_lb_N, "E0": E0_N, "E_ub": E_ub_N}
    window = E_ub_N - E_lb_N
    print(f"{N:>3d}  {E_lb_N:>12.6f}  {E0_N:>12.6f}  {E_ub_N:>12.6f}  {window:>10.6f}")

  N    E_lb (SDP)    E0 (exact)   E_ub (DMRG)      Window
----------------------------------------------------------


/users/eleves-b/2023/sherab-losel.matos/thesis/SpinsSDP/SpinsSDP/.venv/lib/python3.13/site-packages/tenpy/networks/mps.py:1629: UserWarning: unit_cell_width is a new argument for MPS and similar classes. It is optional for now, but will become mandatory in a future release. The default value (unit_cell_width=len(sites)) is correct, iff the lattice is a Chain. For other lattices, it is incorrect. It is used for dipolar charges and correlation_function2.
  super().__init__(sites, bc, unit_cell_width)


  4     -2.000000     -2.000000     -2.000000   -0.000000


/users/eleves-b/2023/sherab-losel.matos/thesis/SpinsSDP/SpinsSDP/.venv/lib/python3.13/site-packages/tenpy/networks/mps.py:1629: UserWarning: unit_cell_width is a new argument for MPS and similar classes. It is optional for now, but will become mandatory in a future release. The default value (unit_cell_width=len(sites)) is correct, iff the lattice is a Chain. For other lattices, it is incorrect. It is used for dipolar charges and correlation_function2.
  super().__init__(sites, bc, unit_cell_width)


  6     -2.802775     -2.802776     -2.802776   -0.000000


...

  8     -3.651708     -3.651093     -3.651093    0.000615


/users/eleves-b/2023/sherab-losel.matos/thesis/SpinsSDP/SpinsSDP/.venv/lib/python3.13/site-packages/tenpy/networks/mps.py:1629: UserWarning: unit_cell_width is a new argument for MPS and similar classes. It is optional for now, but will become mandatory in a future release. The default value (unit_cell_width=len(sites)) is correct, iff the lattice is a Chain. For other lattices, it is incorrect. It is used for dipolar charges and correlation_function2.
  super().__init__(sites, bc, unit_cell_width)


 10     -4.518159     -4.515446     -4.515446    0.002713


/users/eleves-b/2023/sherab-losel.matos/thesis/SpinsSDP/SpinsSDP/.venv/lib/python3.13/site-packages/tenpy/networks/mps.py:1629: UserWarning: unit_cell_width is a new argument for MPS and similar classes. It is optional for now, but will become mandatory in a future release. The default value (unit_cell_width=len(sites)) is correct, iff the lattice is a Chain. For other lattices, it is incorrect. It is used for dipolar charges and correlation_function2.
  super().__init__(sites, bc, unit_cell_width)


 12     -5.392883     -5.387391     -5.387391    0.005492


In [24]:
# Observable bounds scaling: nearest-neighbour ZZ correlation
print(f"\nNearest-neighbour ZZ correlation bounds vs N:\n")
print(f"{'N':>3s}  {'Lower':>10s}  {'Exact':>10s}  {'Upper':>10s}  {'Width':>10s}")
print("-" * 50)

for N in Ns:
    H_dict_N = heisenberg_hamiltonian_dict(N, boundary="periodic")
    sym_N = SymmetryManager.default_for_heisenberg(N)
    basis_N = generate_npa_basis(N, k=2)
    obs_zz_N = nearest_neighbor_correlation_dict(N, axis="z", boundary="periodic")
    
    E_lb_N = energy_data[N]["E_lb"]
    E_ub_N = energy_data[N]["E_ub"]
    
    # Add a small tolerance to handle numerical noise in tight windows
    margin = max(1e-6, abs(E_ub_N - E_lb_N) * 0.01)
    
    res = bound_observable(
        basis=basis_N.words,
        hamiltonian=H_dict_N,
        observable=obs_zz_N,
        energy_lb=E_lb_N - margin,
        energy_ub=E_ub_N + margin,
        symmetry_manager=sym_N,
        mosek_tol=1e-7,
    )
    
    # Exact value
    H_exact_N = heisenberg_hamiltonian_exact(N, boundary="periodic")
    exact_val = exact_expectation(N, obs_zz_N, H_exact_N.groundstate()[1])
    
    print(f"{N:>3d}  {res.lb:>10.6f}  {exact_val:>10.6f}  {res.ub:>10.6f}  {res.ub - res.lb:>10.6f}")


Nearest-neighbour ZZ correlation bounds vs N:

  N       Lower       Exact       Upper       Width
--------------------------------------------------
  4   -0.666939   -0.666667   -0.666394    0.000545
  6   -0.623077   -0.622839   -0.622601    0.000476
  8   -0.613721   -0.608516   -0.603171    0.010550
 10   -0.611965   -0.602060   -0.591641    0.020324
 12   -0.611213   -0.598599   -0.584816    0.026396


---
## 7. Persistence with the ArtifactManager

For production runs, we can save observable bounds atomically using the `ArtifactManager`.

In [25]:
# Set up a demo directory
demo_root = Path("observable_demo_results")
if demo_root.exists():
    shutil.rmtree(demo_root)
demo_root.mkdir()

am = ArtifactManager(demo_root)

# Configuration for this experiment
obs_config = {
    "model": "heisenberg",
    "boundary": "periodic",
    "npa_level": 2,
    "chi_max": 128,
    "observable": "nn_zz_correlation",
}

run = am.create_run(
    artifact="observable_bounds",
    name="heisenberg_nn_zz",
    config=obs_config,
)
print(f"Run directory: {run.path}")

# Compute and save bounds for each N
Ns_arr = np.array([4, 6, 8], dtype=np.int32)
lbs = []
ubs = []
E_lbs = []
E_ubs = []

for N in Ns_arr:
    H_dict_N = heisenberg_hamiltonian_dict(N, boundary="periodic")
    sym_N = SymmetryManager.default_for_heisenberg(N)
    basis_N = generate_npa_basis(N, k=2)
    obs_N = nearest_neighbor_correlation_dict(N, axis="z", boundary="periodic")
    
    e_lb = energy_data[N]["E_lb"]
    e_ub = energy_data[N]["E_ub"]
    margin = max(1e-6, abs(e_ub - e_lb) * 0.01)
    
    res = bound_observable(
        basis=basis_N.words,
        hamiltonian=H_dict_N,
        observable=obs_N,
        energy_lb=e_lb - margin,
        energy_ub=e_ub + margin,
        symmetry_manager=sym_N,
        mosek_tol=1e-7,
    )
    
    print(f"  N={N}: [{res.lb:.6f}, {res.ub:.6f}]")
    lbs.append(res.lb)
    ubs.append(res.ub)
    E_lbs.append(e_lb)
    E_ubs.append(e_ub)

run.save_table(
    N=Ns_arr,
    obs_lb=np.array(lbs),
    obs_ub=np.array(ubs),
    energy_lb=np.array(E_lbs),
    energy_ub=np.array(E_ubs),
)
run.update_meta(status="complete", Ns=[int(n) for n in Ns_arr])
print(f"\nResults saved to {run.path}")

Run directory: observable_demo_results/observable_bounds/v1/heisenberg_nn_zz
  N=4: [-0.666939, -0.666394]
  N=6: [-0.623077, -0.622601]
  N=8: [-0.613721, -0.603171]

Results saved to observable_demo_results/observable_bounds/v1/heisenberg_nn_zz


In [26]:
# Load results back
am2 = ArtifactManager(demo_root)

loaded_run = am2.open_run(artifact="observable_bounds", name="heisenberg_nn_zz")
data = loaded_run.load_table()
meta = loaded_run.load_meta()

print(f"Loaded run: {loaded_run.name}")
print(f"Status: {meta.get('status', 'unknown')}")
print(f"Columns: {sorted(data.keys())}\n")

print(f"{'N':>3s}  {'E_lb':>10s}  {'E_ub':>10s}  {'Obs_lb':>10s}  {'Obs_ub':>10s}")
print("-" * 50)
for i in range(len(data["N"])):
    print(f"{data['N'][i]:>3d}  {data['energy_lb'][i]:>10.6f}  "
          f"{data['energy_ub'][i]:>10.6f}  {data['obs_lb'][i]:>10.6f}  "
          f"{data['obs_ub'][i]:>10.6f}")

Loaded run: heisenberg_nn_zz
Status: complete
Columns: ['N', 'energy_lb', 'energy_ub', 'obs_lb', 'obs_ub']

  N        E_lb        E_ub      Obs_lb      Obs_ub
--------------------------------------------------
  4   -2.000000   -2.000000   -0.666939   -0.666394
  6   -2.802775   -2.802776   -0.623077   -0.622601
  8   -3.651708   -3.651093   -0.613721   -0.603171


---
## Summary

In this notebook we showed how to:

1. **Obtain energy bounds** - SDP relaxation for the lower bound, DMRG for the upper bound.
2. **Bound any observable** - by adding energy constraints to the SDP and solving the min/max.
3. **Verify bounds** - against exact diagonalisation for small systems.
4. **Scale up** - the method works beyond exact diagonalisation limits using DMRG for the upper bound.
5. **Persist results** - atomic checkpointing with the `ArtifactManager`.

The `bound_observable(...)` function encapsulates the entire workflow: given a basis, Hamiltonian, observable, and energy window, it returns certified lower and upper bounds on $\langle O \rangle_{\text{g.s.}}$.